In [14]:
import fitz          # PyMuPDF
import unicodedata
import re
import pandas as pd

def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")

def extract_blocks(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    full = ""
    for page in doc:
        full += page.get_text("text") + "\n\f\n"
    full = strip_accents(full)
    return full.split("FICHA TECNICA")[1:]

def parse_block(block: str) -> dict | None:
    lines = [ln.strip() for ln in strip_accents(block).splitlines() if ln.strip()]
    upper = [ln.upper() for ln in lines]

    # Saltar encabezado de columnas si existe
    headers = ["PROYECTO","RADICACION","TIPO DE PROYECTO","SEUDONIMO"]
    if len(upper)>=4 and all(upper[i]==headers[i] for i in range(4)):
        lines = lines[4:]; upper = upper[4:]

    rec = {}

    # — No. CÁMARA con posible “ACU AL”
    if "NO. CAMARA" in upper:
        i = upper.index("NO. CAMARA")
        if i+1 < len(lines):
            cam_line = lines[i+1]
            # Caso especial ACU AL:
            if "ACU AL" in cam_line.upper():
                left, right = re.split(r"ACU AL", cam_line, flags=re.IGNORECASE)
                m1 = re.search(r"(\d+)/\d{4}[CS]", left)
                m2 = re.search(r"(\d+)/\d{4}[CS]", right)
                if m1 and m2:
                    rec["num_camara"] = m1.group(1)
                    rec["acu_al"]     = m2.group(1)
            else:
                m = re.search(r"(\d+)/\d{4}[CS]", cam_line)
                if m:
                    rec["num_camara"] = m.group(1)

    # — No. SENADO (si existe)
    if "NO. SENADO" in upper:
        j = upper.index("NO. SENADO")
        if j+1 < len(lines):
            sen_line = lines[j+1]
            m = re.search(r"(\d+)/\d{4}[CS]", sen_line)
            if m:
                rec["num_senado"] = m.group(1)

    # — Fecha de radicación
    fecha = next((ln for ln in lines if re.fullmatch(r"\d{2}/\d{2}/\d{4}", ln)), None)
    if not fecha:
        return None
    rec["fecha_radicacion"] = fecha

    # — Tipo proyecto y seudónimo
    idx_f = lines.index(fecha)
    if idx_f+1 < len(lines): rec["tipo_proyecto"] = lines[idx_f+1]
    if idx_f+2 < len(lines): rec["seudonimo"]     = lines[idx_f+2]

    # — Comisión y Cámara de origen
    if "COMISION" in upper:
        k = upper.index("COMISION")
        if k+1 < len(lines): rec["comision"] = lines[k+1]
    if "CAMARA DE ORIGEN" in upper:
        k = upper.index("CAMARA DE ORIGEN")
        if k+1 < len(lines): rec["camara_origen"] = lines[k+1]

    # — Título
    if "TITULO" in upper:
        t0 = upper.index("TITULO")
        buf = []
        for ln in lines[t0+1:]:
            if ln.upper().startswith(("AUTOR","PRIMERA","SEGUNDA","MIEMBROS","PUBLICACIONES","ESTADO")):
                break
            buf.append(ln)
        rec["titulo"] = " ".join(buf)

    # — Autores
    autor_idx = next((idx for idx,up in enumerate(upper) if up.startswith("AUTOR")), None)
    if autor_idx is not None:
        buf = []
        for ln in lines[autor_idx+1:]:
            if ln.upper().startswith(("PRIMERA","SEGUNDA","PUBLICACIONES","ESTADO","PONENTES","MIEMBROS")):
                break
            buf.append(ln)
        rec["autores"] = " ".join(buf)

    return rec


# ——— EJECUCIÓN EN TU NOTEBOOK ———

pdf_path = r"C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2022 2023 LEGISLATURA_proyectos_ley_actos_Legislativos.pdf"
out_xlsx = "fichas_tecnicas.xlsx"

bloques    = extract_blocks(pdf_path)
parsed     = [parse_block(b) for b in bloques]
registros  = [r for r in parsed if r]

df = pd.DataFrame(registros)
print(f"✅ Fichas válidas extraídas: {len(df)}")
display(df.head())

df.to_excel(out_xlsx, index=False)
print("✅ Excel guardado en", out_xlsx)


✅ Fichas válidas extraídas: 328


,num_camara,fecha_radicacion,tipo_proyecto,seudonimo,comision,camara_origen,titulo,autores,num_senado
0,001,21/07/2022,PROYECTO DE LEY ORDINARIA,INDUSTRIA Y COMERCIO,COMISION TERCERA CONSTITUCIONAL PERMANENTE,CAMARA,Por medio del cual el impuesto de industria y ...,H.S.Ana carolina Espitia Jerez H.R.Juan Diego ...,NaN
1,002,21/07/2022,PROYECTO DE ACTO LEGISLATIVO,CANNABIS,COMISION PRIMERA CONSTITUCIONAL PERMANENTE,CAMARA,Por medio del cual se modifica el articulo 49 ...,"H.S.Alejandro Alberto Vega Perez , H.S.Alejand...",033
2,003,21/07/2022,PROYECTO DE ACTO LEGISLATIVO,DERECHOS DE LA NATURALEZA,COMISION PRIMERA CONSTITUCIONAL PERMANENTE,CAMARA,Por el cual se modifican los articulos 79 y 95...,"H.S.Alejandro Alberto Vega Perez , H.S.Nicolas...",NaN
3,004,21/07/2022,PROYECTO DE ACTO LEGISLATIVO,SEMILLAS TRANSGENICAS,COMISION PRIMERA CONSTITUCIONAL PERMANENTE,CAMARA,Por medio del cual se modifica el articulo 81 ...,"H.S.Alejandro Alberto Vega Perez , H.S.Edwing ...",NaN
4,005,21/07/2022,PROYECTO DE ACTO LEGISLATIVO,ALIMENTACION,COMISION PRIMERA CONSTITUCIONAL PERMANENTE,CAMARA,Por el cual se modifica el articulo 65 de la c...,"H.S.Alejandro Alberto Vega Perez , H.S.Edwing ...",NaN


✅ Excel guardado en fichas_tecnicas.xlsx
